# Heart Disease Prediction using Support Vector Classifier (SVC)

**Live Demo (Streamlit):** https://YOUR-APP-NAME.streamlit.app/
**GitHub Repo:** https://github.com/YOUR-USERNAME/heart-disease-app

**Dataset:** UCI Heart Disease (Cleveland) - https://archive.ics.uci.edu/dataset/45/heart+disease

**Selected 4 input attributes:** Age, Sex, ChestPain (cp), MaxHeartRate (thalach)

**Coding (already numeric in the UCI data, so no label encoding is needed):**
Sex: 1 = Male, 0 = Female | ChestPain: 1 = Typical Angina, 2 = Atypical Angina, 3 = Non-anginal Pain, 4 = Asymptomatic
**Target:** HeartDisease (0 = No disease, 1 = Disease present)

In [ ]:
"""
Purpose of this section:
-------------------------
Import all the libraries needed for loading data, encoding, training,
evaluating and saving the model.
"""
import pickle
import warnings
import subprocess, sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

try:
    from prettytable import PrettyTable
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "prettytable"])
    from prettytable import PrettyTable

warnings.filterwarnings("ignore")
print("Libraries imported successfully")

In [ ]:
"""
Purpose of this section:
-------------------------
Load the dataset (heart.csv). It contains only the 4 selected input
attributes plus the target column. Run prepare_dataset.py once to create it
from the UCI Cleveland file.
"""
dataset = pd.read_csv("heart.csv")

print("Dataset shape:", dataset.shape)
print()
print(dataset.head())
print()
print("Missing values per column:")
print(dataset.isnull().sum())

In [ ]:
"""
Purpose of this section:
-------------------------
Exploratory Data Analysis (EDA): summary statistics, class balance and a few
plots to understand the data before modelling.
"""
print(dataset.describe(include="all"))
print()
print("Class distribution (HeartDisease):")
print(dataset["HeartDisease"].value_counts())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.countplot(x="HeartDisease", data=dataset, ax=axes[0])
axes[0].set_title("Target distribution")
sns.histplot(data=dataset, x="Age", hue="HeartDisease", kde=True, ax=axes[1])
axes[1].set_title("Age vs Heart Disease")
sns.countplot(x="ChestPain", hue="HeartDisease", data=dataset, ax=axes[2])
axes[2].set_title("Chest pain type vs Heart Disease")
axes[2].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
"""
Purpose of this section:
-------------------------
Split the data into a training set (80%) and a testing set (20%).
Stratify keeps the same class ratio in both sets.
"""
training_data, testing_data = train_test_split(
    dataset, test_size=0.2, random_state=42, stratify=dataset["HeartDisease"]
)

training_data.to_csv("training-data.csv", index=False)
testing_data.to_csv("testing-data.csv", index=False)

print("Training data shape:", training_data.shape)
print("Testing data shape:", testing_data.shape)

In [ ]:
"""
Purpose of this section:
-------------------------
All 4 selected attributes are already numeric in the UCI dataset
(Sex: 1/0, ChestPain: 1-4), so no label encoding is required.
We only verify the data types here.
"""
print(training_data.dtypes)
print()
print(training_data.head())

In [ ]:
"""
Purpose of this section:
-------------------------
This part of the code separates the features (inputs) and the labels (outputs)
from the training dataset.
"""

input_vector_train = training_data.iloc[:, :-1]
output_label_train = training_data.iloc[:, -1]

print("Input vectors (train) shape:", input_vector_train.shape)
print("Output labels (train) shape:", output_label_train.shape)

In [ ]:
"""
Purpose of this section:
-------------------------
Compare a few classifiers using 5-fold cross-validation on the training data.
Age and MaxHeartRate are on different scales, so every model is wrapped in a
pipeline with StandardScaler.
"""
from sklearn.model_selection import cross_val_score

candidates = {
    "SVC": Pipeline([("scaler", StandardScaler()), ("clf", SVC(kernel="rbf", C=1.0))]),
    "Logistic Regression": Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression())]),
    "Decision Tree": Pipeline([("scaler", StandardScaler()), ("clf", DecisionTreeClassifier(max_depth=4, random_state=42))]),
}

for name, candidate in candidates.items():
    scores = cross_val_score(candidate, input_vector_train, output_label_train, cv=5)
    print(f"{name:20s} CV accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

In [ ]:
"""
Purpose of this section:
-------------------------
This part of the code trains the Support Vector Classifier (SVC) on the
training data and saves the trained model (with its scaler) to disk.
"""
svc_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", C=1.0, probability=True, random_state=42)),
])
svc_model.fit(input_vector_train, output_label_train)

pickle.dump(svc_model, open("svc_trained_model.pkl", "wb"))
print("Model saved to svc_trained_model.pkl")

In [ ]:
"""
Purpose of this section:
-------------------------
This part of the code separates the features (inputs) and the labels (outputs)
from the testing dataset.
"""

input_vector_test = testing_data.iloc[:, :-1]
output_label_test = testing_data.iloc[:, -1]

print("Input vectors (test) shape:", input_vector_test.shape)
print("Output labels (test) shape:", output_label_test.shape)

In [ ]:
"""
Purpose of this section:
-------------------------
This part of the code loads the previously saved Support Vector Classifier (SVC)
model from disk into memory.
"""

model = pickle.load(open('svc_trained_model.pkl', 'rb'))
print("Model loaded from svc_trained_model.pkl")

In [ ]:
"""
Purpose of this section:
-------------------------
This part of the code evaluates the trained Support Vector Classifier (SVC)
on the testing data.
"""

model_predictions = model.predict(input_vector_test)

pd.options.mode.chained_assignment = None
testing_data["Predictions"] = model_predictions
testing_data.to_csv(r'model-predictions.csv', index=False, header=True)

model_predictions = testing_data
print("\n\nPredictions Returned by svc_trained_model:")
print("==========================================\n")
print(model_predictions.head(15))

In [ ]:
"""
Purpose of this section:
-------------------------
This part of the code calculates how accurate the trained model is on the
testing dataset.
"""

model_accuracy_score = accuracy_score(
    model_predictions["HeartDisease"],
    model_predictions["Predictions"]
)

print("\n\nAccuracy Score:")
print("===============\n")
print(round(model_accuracy_score, 2))

In [ ]:
"""
Purpose of this section:
-------------------------
Accuracy alone can be misleading. This section adds the standard
classification metrics:
- Precision: of everyone predicted as "Disease", how many really have it?
- Recall: of everyone who really has disease, how many did the model catch?
- F1-score: the harmonic mean of precision and recall.
- Confusion Matrix: a table of correct vs. incorrect predictions per class.
For a medical problem, recall on the "Disease" class matters a lot because a
missed case is costlier than a false alarm.
"""

print("\nClassification Report:")
print("=======================\n")
print(classification_report(
    model_predictions["HeartDisease"],
    model_predictions["Predictions"],
    target_names=["No Disease (0)", "Disease (1)"]
))

conf_matrix = confusion_matrix(model_predictions["HeartDisease"], model_predictions["Predictions"])
conf_matrix_df = pd.DataFrame(
    conf_matrix,
    index=["Actual: No Disease", "Actual: Disease"],
    columns=["Predicted: No Disease", "Predicted: Disease"]
)
print("\nConfusion Matrix:")
print("==================\n")
print(conf_matrix_df)

plt.figure(figsize=(5, 4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - SVC on Heart Disease Testing Data')
plt.tight_layout()
plt.show()

In [ ]:
"""
Purpose of this section:
-------------------------
This part of the code allows the user to enter their own data
(Age, Sex, ChestPain, MaxHeartRate). In an interactive Jupyter session,
running this cell will prompt you for input(). For this notebook copy,
we pre-fill a demo patient so the notebook can also be read/run
top-to-bottom without needing to type anything.

To try your own values interactively, replace the four lines below with:
    age_input = int(input("Please enter Age here (e.g. 54): "))
    sex_input = int(input("Please enter your Sex here (1 = Male, 0 = Female): "))
    chestpain_input = int(input("Please enter ChestPain here (1 = Typical Angina, 2 = Atypical Angina, 3 = Non-anginal Pain, 4 = Asymptomatic): "))
    maxhr_input = int(input("Please enter MaxHeartRate here (e.g. 150): "))
"""

# Demo patient (edit these values, or swap in input() calls as shown above)
age_input = 58
sex_input = 1  # 1 = Male, 0 = Female
chestpain_input = 4  # 1 Typical, 2 Atypical, 3 Non-anginal, 4 Asymptomatic
maxhr_input = 120

print(f"Age: {age_input}, Sex: {sex_input}, ChestPain: {chestpain_input}, MaxHeartRate: {maxhr_input}")

In [ ]:
"""
Purpose of this section:
-------------------------
This part of the code takes the user's input (entered as text/numbers)
and converts it into a structured feature vector (DataFrame).
"""

user_input = pd.DataFrame({
    'Age': [age_input],
    'Sex': [sex_input],
    'ChestPain': [chestpain_input],
    'MaxHeartRate': [maxhr_input]
})

print("\n\nUser Input Feature Vector:")
print("==========================\n")
print(user_input)

In [ ]:
"""
Purpose of this section:
-------------------------
The user's input is already numeric (Sex: 1/0, ChestPain: 1-4), so it can be
passed to the model directly. No encoder is needed.
"""

unseen_data_features = user_input.copy()

print("\n\nUser Input Feature Vector (model-ready):")
print("=====================================\n")
print(unseen_data_features)

In [ ]:
"""
Purpose of this section:
-------------------------
This part of the code loads the previously saved Support Vector Classifier (SVC)
model from the .pkl file into memory for the Application Phase.
"""

model = pickle.load(open('svc_trained_model.pkl', 'rb'))

In [ ]:
"""
Purpose of this section:
-------------------------
This part of the code uses the trained Support Vector Classifier (SVC)
to predict whether a new patient (user input) has heart disease or not.
"""

predicted_disease = model.predict(unseen_data_features)[0]
probability = model.predict_proba(unseen_data_features)[0][1]

if predicted_disease == 1:
    prediction = "HEART DISEASE LIKELY"
else:
    prediction = "NO HEART DISEASE LIKELY"

pretty_table = PrettyTable()
pretty_table.add_column("       ** Prediction **       ", [prediction])
pretty_table.add_column("   Disease probability   ", [f"{probability:.0%}"])

print(pretty_table)
print("Educational project only - not medical advice.")

In [ ]:
"""
Purpose of this section:
-------------------------
Step 9.1: Feedback collected from reviewers on a few test cases.
NOTE: Replace the rows below with feedback from real people you asked
(a classmate, a family member in the medical field, etc.) and the real
predictions your model gave them.
"""

example_feedback = pd.DataFrame({
    "Reviewer": ["Student Tester", "Student Tester", "Domain Reviewer"],
    "Case": [
        "Age 58, Sex 1 (Male), ChestPain 4 (Asymptomatic), MaxHR 120 -> predicted HEART DISEASE LIKELY",
        "Age 35, Sex 0 (Female), ChestPain 3 (Non-anginal), MaxHR 175 -> predicted NO HEART DISEASE LIKELY",
        "Age 62, Sex 1 (Male), ChestPain 1 (Typical Angina), MaxHR 165 -> prediction to be checked",
    ],
    "Feedback": [
        "Plausible: asymptomatic chest pain, older age and low max heart rate are known risk signs in this dataset.",
        "Plausible: young age and high max heart rate are linked with lower risk.",
        "Only 4 features are used, so important factors (cholesterol, ECG, exercise angina) are missing.",
    ],
})
print(example_feedback.to_string(index=False))

In [ ]:
"""
Purpose of this section:
-------------------------
Step 9.2: Potential improvements identified from the feedback above.
"""

improvements = [
    "Add more attributes (e.g. cholesterol, resting blood pressure, exercise-induced angina, ST depression) - the task limits us to 4, but a real system would use more",
    "Combine the Cleveland data with the other UCI sites (Hungary, Switzerland, VA) for more rows",
    "ChestPain is a nominal category (1-4 are labels, not an order); one-hot encoding it may improve results",
    "Tune SVC hyperparameters (C, gamma, kernel) with GridSearchCV",
    "Compare additional models (e.g. Random Forest, Gradient Boosting) and prefer high recall on the disease class",
]
for i, item in enumerate(improvements, 1):
    print(f"{i}. {item}")

In [ ]:
"""
Purpose of this section:
-------------------------
Step 9.3: In a real deployment, the next version of this notebook would
retrain on the improvements listed above and re-run the training and
evaluation steps to confirm the new model improves before replacing the
deployed model.
"""

print("Next version: retrain with tuned hyperparameters / extra features, then re-evaluate before replacing svc_trained_model.pkl")